# LSTM tuning

Using PCA to determine the internal (hidden) dimension of the LSTM portion of SHRED for this use case

Sources:
1. DB's coursework solution
2. [Reducing dimensionality with principal component analysis with Python](https://developer.ibm.com/tutorials/awb-reducing-dimensionality-with-principal-component-analysis/)

In [ ]:
import json

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

# Local imports
from helper_functions.ofam3_data import load_files
from helper_functions.preprocess_SHRED import (
    convert_to_anomaly,
    split_ordered_data,
)
from helper_functions.saving import JsonEncoder

In [ ]:
USE_ANOMALIES = True

In [ ]:
data, dates, lats, longs = load_files()

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, test_dates = split_ordered_data(data, dates)

if USE_ANOMALIES:
    train_data, val_data, test_data, bulk_mean = convert_to_anomaly((train_data, val_data, test_data))

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
train_transformed = sc.transform(train_data)  
val_transformed = sc.transform(val_data)
test_transformed = sc.transform(test_data)

In [ ]:
train_transformed.shape

In [ ]:
# Plot of data over time
colours = plt.cm.jet(np.linspace(0,1,train_data.shape[1]))
for i in range(0, train_data.shape[1]):  # plot first 10 sensor locations
    data = np.array([data_at_time[i] for data_at_time in train_data])
    plt.plot(data, c=colours[i])
plt.xlabel("Timesteps")
plt.ylabel("SST ($\\degree$C)")
plt.show()

In [ ]:
# Plot of data over time
for i in range(train_data.shape[1]):  # plot first 10 sensor locations
    data = np.array([data_at_time[i] for data_at_time in train_transformed])
    plt.plot(data, c=colours[i])
plt.xlabel("Timesteps")
plt.ylabel("Normalised SST ($\\degree$C)")
plt.show()

In [ ]:
latent_dim = []
# Use PCA on many time series to find shared temporal patterns across them
for idx in range(1, train_transformed.shape[0]):
    # data.shape needs to be (rows, columns) = (n_samples, n_features)
    # rows = samples (e.g., time points), columns = features (e.g., individual time series) <- values at each time point for each sensor

    # Crop for the number of timesteps, then transform into timeseries sequences for each sensor
    X = train_transformed[:idx]

    # Center data: mean subtract each data
    Xc = X - X.mean(axis=0)

    # SVD:
    # # U: (n_samples, k), S: (k,), Vt: (k, n_features), where k = min(n_samples, n_features)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)  # SVD

    # Eigenvalues of covariance matrix
    n = X.shape[0]
    eigenvals = (S**2) / (n - 1)

    # Explained variance and ratio
    explained_variance = eigenvals
    explained_variance_ratio = explained_variance / explained_variance.sum()
    cumulative_explained = np.cumsum(explained_variance_ratio)
    dim = np.argmax(cumulative_explained >= 0.995) + 1 # +1 as indices star
    latent_dim.append(dim)
plt.plot(latent_dim, c="#1589e8")
plt.grid()
plt.xlabel("Time step")
plt.ylabel("Dimensions required to explain 99.5% of variance")
plt.show()

## Reduced number of sensors

In [ ]:
num_sensors = 1059
sensor_locations = np.random.choice(train_data.shape[1], size=num_sensors, replace=False)

In [ ]:
# Plot of data over time
data_for_sensors = train_data[:, sensor_locations]
for i in range(num_sensors):
    plt.plot(data_for_sensors.T[i])

plt.xlabel("Timesteps")
plt.ylabel("SST ($\\degree$C)")
plt.show()

In [ ]:
# Plot of data over time
sensor_data_transf = train_transformed[:, sensor_locations]
for i in range(num_sensors):
    plt.plot(sensor_data_transf.T[i])

plt.xlabel("Timesteps")
plt.ylabel("Normalised SST ($\\degree$C)")
plt.show()

In [ ]:
sensor_data_transf.shape

### All 59063 locations

In [ ]:
latent_dim = []
# Use PCA on many time series to find shared temporal patterns across them
for idx in range(1, sensor_data_transf.shape[0]):
    # data.shape needs to be (rows, columns) = (n_samples, n_features)
    # rows = samples (e.g., time points), columns = features (e.g., individual time series) <- values at each time point for each sensor

    # Crop for the number of timesteps, then transform into timeseries sequences for each sensor
    X = sensor_data_transf[:idx]

    # Center data: mean subtract each data
    Xc = X - X.mean(axis=0)

    # SVD:
    # # U: (n_samples, k), S: (k,), Vt: (k, n_features), where k = min(n_samples, n_features)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)  # SVD

    # Eigenvalues of covariance matrix
    n = X.shape[0]
    eigenvals = (S**2) / (n - 1)

    # Explained variance and ratio
    explained_variance = eigenvals
    explained_variance_ratio = explained_variance / explained_variance.sum()
    cumulative_explained = np.cumsum(explained_variance_ratio)
    dim = np.argmax(cumulative_explained >= 0.995) + 1 # +1 as indices star
    latent_dim.append(dim)

In [ ]:
plt.plot(latent_dim, c="#1589e8")
plt.grid()
plt.xlabel("Time step")
plt.ylabel("Dimensions required to explain 99.5% of variance")
plt.show()

In [ ]:
with open('Data/pca_data_59063.json', 'w') as f:
    json.dump(latent_dim, f, cls=JsonEncoder)

### Varying sensor numbers:

- 5-145 sensors:
    - options = list(range(5, 165, 20))
    - file_name = "pca_data_5_145.json"
- 5-1955 sensors: options = list(range(5, 2000, 150))

In [ ]:
options = list(range(5, 2000, 150))
file_name = "pca_anomalies_5_1955.json"
options

In [ ]:
latent_dim_dict = {}
for num_sensors in tqdm(options):
    sensor_locations = np.random.choice(train_data.shape[1], size=num_sensors, replace=False)
    sensor_data_transf = train_transformed[:, sensor_locations]

    latent_dim = []
    # Use PCA on many time series to find shared temporal patterns across them
    for idx in range(1, sensor_data_transf.shape[0]):
        # data.shape needs to be (rows, columns) = (n_samples, n_features)
        # rows = samples (e.g., time points), columns = features (e.g., individual time series) <- values at each time point for each sensor

        # Crop for the number of timesteps, then transform into timeseries sequences for each sensor
        X = sensor_data_transf[:idx]

        # Center data: mean subtract each data
        Xc = X - X.mean(axis=0)

        # SVD:
        # # U: (n_samples, k), S: (k,), Vt: (k, n_features), where k = min(n_samples, n_features)
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)  # SVD

        # Eigenvalues of covariance matrix
        n = X.shape[0]
        eigenvals = (S**2) / (n - 1)

        # Explained variance and ratio
        explained_variance = eigenvals
        explained_variance_ratio = explained_variance / explained_variance.sum()
        cumulative_explained = np.cumsum(explained_variance_ratio)
        dim = np.argmax(cumulative_explained >= 0.995) + 1 # +1 as indices star
        latent_dim.append(dim)

    latent_dim_dict[num_sensors] = latent_dim

In [ ]:
colours = plt.cm.jet(np.linspace(0,1,len(options)))
for i, (num_sensors, v) in enumerate(latent_dim_dict.items()):
    plt.plot(v, c=colours[i], label=num_sensors)

plt.grid()
plt.axvline(52, c='k', ls='--')
plt.xlabel("Time step")
plt.ylabel("Dimensions required to explain 99.5% of variance")
plt.legend(bbox_to_anchor=(1,1), title="Number of sensors")
plt.show()

In [ ]:
colours = plt.cm.jet(np.linspace(0,1,len(options)))
for i, (num_sensors, v) in enumerate(latent_dim_dict.items()):
    plt.plot(v, c=colours[i], label=num_sensors)

plt.grid()
plt.axvline(52, c='k', ls='--')
plt.xlabel("Time step")
plt.ylabel("Dimensions required to explain 99.5% of variance")
plt.legend(bbox_to_anchor=(1, 1), title="Number of sensors")

plt.xlim(0, 350)
plt.ylim(0, 60)
plt.show()

In [ ]:
with open(f"Data/{file_name}.json", 'w') as f:
    json.dump(latent_dim_dict, f, cls=JsonEncoder)

### 5-1955 sensors

In [ ]:
with open('Data/pca_data_5_1955.json', 'r') as f:
    data = json.load(f)

In [ ]:
colours = plt.cm.jet(np.linspace(0,1,len(data)))
for i, (num_sensors, v) in enumerate(data.items()):
    plt.plot(v, c=colours[i], label=num_sensors)

plt.grid()
plt.axvline(52, c='k', ls='--')
plt.xlabel("Time step")
plt.ylabel("Dimensions required to explain 99.5% of variance")
plt.legend(bbox_to_anchor=(1,1), title="Number of sensors")
plt.show()